In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (결측치 원천 차단)
# ==========================================================
weather_cols = ['평균기온(°C)', '최고기온(°C)', '최저기온(°C)', '평균 상대습도(%)', 
                '일강수량(mm)', '합계 일조시간(hr)', '평균 풍속(m/s)', '평균 이슬점온도(°C)']

valid_cols = [c for c in weather_cols if c in weather_df.columns]

# 💡 핵심 수정: 결측치가 하나라도 있는 날짜의 기상 데이터는 아예 빼버림 (왜곡 원천 차단)
weather_df = weather_df.dropna(subset=valid_cols)
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)

wg = weather_df.groupby('매칭지역')

# 살아남은 깨끗한 데이터들로만 14일 치 롤링 계산
weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)
weather_df['강수량_sum']      = wg['일강수량(mm)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)
weather_df['일조시간_sum']    = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '강수량_sum', '일조시간_sum', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 및 엄격한 데이터 필터링
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ 결측치 제거 후 순수 데이터 모델 스캔 중... (엄격한 테스트 검증 적용)\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    positive_damage = df_subset[df_subset[target] > 0][target]
    if len(positive_damage) == 0: continue 
    
    median_val = positive_damage.median()
    
    def grade_by_median(rate):
        if rate == 0: return 0
        elif rate <= median_val: return 1
        else: return 2
        
    df_subset['damage_grade'] = df_subset[target].apply(grade_by_median)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    # 평가 조건 컷오프 (0, 1, 2 등급이 모두 존재 & 가장 적은 등급이 최소 5개 이상 보장)
    class_counts = merged_df['damage_grade'].value_counts()
    if len(class_counts) < 3 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['damage_grade']
    
    # 층화 추출 강제. 실패 시 무작위 분할 대신 과감히 패스
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
        "XGB": XGBClassifier(random_state=42, eval_metric='mlogloss')
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1, 2], average=None, zero_division=0) * 100
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터수': len(merged_df),
            '중앙값(구분값)': round(median_val, 2),
            '모델': model_name,
            '정확도': round(acc, 1),
            '정상(0) 재현율': round(recalls[0], 1),
            '경미(1) 재현율': round(recalls[1], 1),
            '심각(2) 재현율': round(recalls[2], 1)
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 순수 기상 데이터(결측치 제거) 기반 '진짜' 모델 성능 🔥")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '심각(2) 재현율'], ascending=[True, False])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(결측치 제외 후 각 등급 데이터 최소 5개 이상)을 충족하는 병해충 데이터가 없습니다.")
print("-" * 105)

⏳ 결측치 제거 후 순수 데이터 모델 스캔 중... (엄격한 테스트 검증 적용)

🔥 검증 완료: 순수 기상 데이터(결측치 제거) 기반 '진짜' 모델 성능 🔥
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터수  중앙값(구분값)  모델  정확도  정상(0) 재현율  경미(1) 재현율  심각(2) 재현율
   깨씨무늬병     373      0.01 KNN 45.3       40.0       42.9       61.1
   깨씨무늬병     373      0.01  LR 52.0       52.0       71.4       44.4
   깨씨무늬병     373      0.01  RF 56.0       66.0       57.1       27.8
   깨씨무늬병     373      0.01 XGB 60.0       80.0       14.3       22.2
   끝동매미충     362      0.01  LR 64.4       67.2       50.0       25.0
   끝동매미충     362      0.01 KNN 50.7       50.7      100.0       25.0
   끝동매미충     362      0.01  RF 79.5       83.6       50.0       25.0
   끝동매미충     362      0.01 XGB 84.9       89.6       50.0       25.0
    먹노린재     383      0.02 XGB 49.4       54.8        8.3       60.9
    먹노린재     383      0.02  LR 44.2       40.5       41.7       52.2
    먹노린재     383      0.02  RF 48.1       59.5

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (결측치 관련 쓸데없는 로직 전부 제거)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)
weather_df['강수량_sum']      = wg['일강수량(mm)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)
weather_df['일조시간_sum']    = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).sum().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '강수량_sum', '일조시간_sum', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ 13개 파생변수 모델 스캔 중... (순수 데이터 기준)\n")

for target in target_diseases:
    # 1. 예찰 데이터에 빈칸이 있는 경우만 제거 (정답이 없는 날짜)
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    positive_damage = df_subset[df_subset[target] > 0][target]
    if len(positive_damage) == 0: continue 
    
    median_val = positive_damage.median()
    
    def grade_by_median(rate):
        if rate == 0: return 0
        elif rate <= median_val: return 1
        else: return 2
        
    df_subset['damage_grade'] = df_subset[target].apply(grade_by_median)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    # 🌟 [해결책] 테스트 세트에 정답이 무조건 들어가도록 강제하는 필터
    class_counts = merged_df['damage_grade'].value_counts()
    
    # 클래스(0, 1, 2) 중 하나라도 5개 미만이면 분할 시 테스트 셋에 안 들어갈 수 있으므로 제외
    if len(class_counts) < 3 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['damage_grade']
    
    # stratify=y 를 사용하여 각 등급의 비율을 8:2로 정확히 쪼개도록 강제함.
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        # 쪼개는데 실패하면 억지로 무작위 분할하지 않고 해당 병해충은 버림!
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용 (학습 데이터에만 적용됨)
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
        "XGB": XGBClassifier(random_state=42, eval_metric='mlogloss')
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled) # 한 번도 보지 못한 순수 테스트 셋으로 평가
        
        acc = accuracy_score(y_test, y_pred) * 100
        # 평가할 때 0, 1, 2 라벨을 명시하여 계산 오류 방지
        recalls = recall_score(y_test, y_pred, labels=[0, 1, 2], average=None, zero_division=0) * 100
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터수': len(merged_df),
            '중앙값(구분값)': round(median_val, 2),
            '모델': model_name,
            '정확도': round(acc, 1),
            '정상(0) 재현율': round(recalls[0], 1),
            '경미(1) 재현율': round(recalls[1], 1),
            '심각(2) 재현율': round(recalls[2], 1)
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 파생변수 13개 + 엄격한 층화추출(Stratify) 적용 결과 🔥")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '심각(2) 재현율'], ascending=[True, False])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0,1,2 등급 데이터가 각각 최소 5개 이상)을 충족하는 병해충이 없습니다.")
    print("   이 경우, 3등급 분류는 포기하고 '발생 여부(0, 1)' 이진 분류로 넘어가야 합니다.")
print("-" * 105)

⏳ 13개 파생변수 모델 스캔 중... (순수 데이터 기준)

🔥 검증 완료: 파생변수 13개 + 엄격한 층화추출(Stratify) 적용 결과 🔥
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터수  중앙값(구분값)  모델  정확도  정상(0) 재현율  경미(1) 재현율  심각(2) 재현율
   깨씨무늬병     373      0.01 KNN 45.3       40.0       42.9       61.1
   깨씨무늬병     373      0.01  LR 52.0       52.0       71.4       44.4
   깨씨무늬병     373      0.01  RF 56.0       66.0       57.1       27.8
   깨씨무늬병     373      0.01 XGB 60.0       80.0       14.3       22.2
   끝동매미충     362      0.01  LR 64.4       67.2       50.0       25.0
   끝동매미충     362      0.01 KNN 50.7       50.7      100.0       25.0
   끝동매미충     362      0.01  RF 79.5       83.6       50.0       25.0
   끝동매미충     362      0.01 XGB 84.9       89.6       50.0       25.0
    먹노린재     383      0.02 XGB 49.4       54.8        8.3       60.9
    먹노린재     383      0.02  LR 44.2       40.5       41.7       52.2
    먹노린재     383      0.02  RF 48.1       59.5       

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (중복 sum 제거 -> 총 11개 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

# 강수량_sum, 일조시간_sum 제외
feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (이진 분류 적용)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [최적화 완료] 11개 핵심 피처 + 발생(1) vs 미발생(0) 모델 스캔 중...\n")

for target in target_diseases:
    # 예찰 데이터 결측치 제거
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    # 🌟 0 초과면 무조건 1(발생), 0이면 0(미발생)으로 이진화
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    # 🌟 테스트 세트 보장을 위한 컷오프
    class_counts = merged_df['target_bin'].value_counts()
    
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용 (학습 데이터 50:50 맞춤)
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    # 모델에 가중치(class_weight, scale_pos_weight) 부여하여 발생(1) 재현율 극대화
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        # 이진 분류(0, 1) 재현율 산출
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터수': len(merged_df),
            '모델': model_name,
            '정확도': round(acc, 1),
            '미발생(0) 재현율': round(recalls[0], 1),
            '발생(1) 재현율': round(recalls[1], 1)
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 중복 피처 제거 (11개) + 이진 분류(발생/미발생) 성능 평가 🔥")
print("-" * 90)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '발생(1) 재현율'], ascending=[True, False])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 5개 이상)을 충족하는 병해충이 없습니다.")
print("-" * 90)

⏳ [최적화 완료] 11개 핵심 피처 + 발생(1) vs 미발생(0) 모델 스캔 중...

🔥 검증 완료: 중복 피처 제거 (11개) + 이진 분류(발생/미발생) 성능 평가 🔥
------------------------------------------------------------------------------------------
    병해충명  총 데이터수  모델  정확도  미발생(0) 재현율  발생(1) 재현율
   깨씨무늬병     373 KNN 65.3        65.3       65.4
   깨씨무늬병     373  LR 61.3        61.2       61.5
   깨씨무늬병     373  RF 70.7        77.6       57.7
   깨씨무늬병     373 XGB 61.3        73.5       38.5
   끝동매미충     362  LR 69.9        73.1       33.3
   끝동매미충     362 KNN 60.3        64.2       16.7
   끝동매미충     362  RF 82.2        89.6        0.0
   끝동매미충     362 XGB 82.2        89.6        0.0
    먹노린재     383 XGB 53.2        40.5       68.6
    먹노린재     383  RF 58.4        54.8       62.9
    먹노린재     383 KNN 59.7        64.3       54.3
    먹노린재     383  LR 53.2        61.9       42.9
     벼멸구     365  LR 52.1        45.8       78.6
     벼멸구     365 KNN 63.0        62.7       64.3
     벼멸구     365  RF 71.2        79.7       35.7
     벼멸구     365 XGB 71.2 

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (이진 분류 적용)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ 테스트 세트 비중 확대 (30%) 반영하여 모델 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    # 0 초과면 무조건 1(발생), 0이면 0(미발생)
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    # 🌟 [변경점] test_size를 0.2 -> 0.3으로 확대하여 검증의 신뢰도 향상
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터수': len(merged_df),
            '모델': model_name,
            '정확도': round(acc, 1),
            '미발생(0) 재현율': round(recalls[0], 1),
            '발생(1) 재현율': round(recalls[1], 1)
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 테스트 30% + 11개 피처 + 이진 분류(발생/미발생) 성능 평가 🔥")
print("-" * 90)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '발생(1) 재현율'], ascending=[True, False])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 5개 이상)을 충족하는 병해충이 없습니다.")
print("-" * 90)

⏳ 테스트 세트 비중 확대 (30%) 반영하여 모델 스캔 중...

🔥 검증 완료: 테스트 30% + 11개 피처 + 이진 분류(발생/미발생) 성능 평가 🔥
------------------------------------------------------------------------------------------
    병해충명  총 데이터수  모델  정확도  미발생(0) 재현율  발생(1) 재현율
   깨씨무늬병     373  LR 65.2        62.2       71.1
   깨씨무늬병     373 KNN 55.4        52.7       60.5
   깨씨무늬병     373  RF 65.2        68.9       57.9
   깨씨무늬병     373 XGB 60.7        64.9       52.6
   끝동매미충     362 KNN 65.1        67.0       44.4
   끝동매미충     362  LR 71.6        75.0       33.3
   끝동매미충     362 XGB 82.6        87.0       33.3
   끝동매미충     362  RF 86.2        92.0       22.2
    먹노린재     383 XGB 60.0        53.2       67.9
    먹노린재     383 KNN 64.3        74.2       52.8
    먹노린재     383  RF 56.5        62.9       49.1
    먹노린재     383  LR 56.5        64.5       47.2
     벼멸구     365  LR 48.2        44.9       61.9
     벼멸구     365 KNN 58.2        58.4       57.1
     벼멸구     365  RF 71.8        83.1       23.8
     벼멸구     365 XGB 62.7        71.9

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (이진 분류 적용)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ 테스트 세트 30% 확대 및 발생 건수 정밀 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    # 0 초과면 무조건 1(발생), 0이면 0(미발생)
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    # 🌟 검증 신뢰도를 위해 테스트 세트를 30%로 분할
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTE 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        # 🌟 실제 테스트 데이터 내의 정답 개수와 맞춘 개수 계산
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] # 정렬용 숨김 컬럼
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 검증 완료: 모델별 타격 적중률(맞춘개수/실제개수) 상세 리포트 🔥")
print("-" * 100)
if len(results_df) > 0:
    # 발생(1) 재현율이 높은 순서대로 깔끔하게 정렬
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) # 정렬용 숨김 컬럼 제거
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 5개 이상)을 충족하는 병해충이 없습니다.")
print("-" * 100)

⏳ 테스트 세트 30% 확대 및 발생 건수 정밀 스캔 중...

🔥 검증 완료: 모델별 타격 적중률(맞춘개수/실제개수) 상세 리포트 🔥
----------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도      미발생(0) 방어력     발생(1) 적중률
   깨씨무늬병    373  LR 65.2%   62.2% (46/74) 71.1% (27/38)
   깨씨무늬병    373 KNN 55.4%   52.7% (39/74) 60.5% (23/38)
   깨씨무늬병    373  RF 65.2%   68.9% (51/74) 57.9% (22/38)
   깨씨무늬병    373 XGB 60.7%   64.9% (48/74) 52.6% (20/38)
   끝동매미충    362 KNN 65.1%  67.0% (67/100)   44.4% (4/9)
   끝동매미충    362  LR 71.6%  75.0% (75/100)   33.3% (3/9)
   끝동매미충    362 XGB 82.6%  87.0% (87/100)   33.3% (3/9)
   끝동매미충    362  RF 86.2%  92.0% (92/100)   22.2% (2/9)
    먹노린재    383 XGB 60.0%   53.2% (33/62) 67.9% (36/53)
    먹노린재    383 KNN 64.3%   74.2% (46/62) 52.8% (28/53)
    먹노린재    383  RF 56.5%   62.9% (39/62) 49.1% (26/53)
    먹노린재    383  LR 56.5%   64.5% (40/62) 47.2% (25/53)
     벼멸구    365  LR 48.2%   44.9% (40/89) 61.9% (13/21)
     벼멸구    365 KNN 58.2%   58.4% (52/8

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek  # 🌟 SMOTETomek 추가
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (SMOTETomek + 임계값 30% 적용)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [마스터 레벨] SMOTETomek & 임계값 30% 방어 모델 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌟 [업그레이드 1] SMOTETomek 적용 (경계선 노이즈 제거)
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            # Tomek Links가 데이터가 너무 적을 때 에러날 수 있으므로 예외 처리
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            # 실패 시 일반 SMOTE로 백업
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        
        # 🌟 [업그레이드 2] 확률 기반 예측 (Threshold = 0.3)
        # 발생(1) 클래스일 확률만 추출
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        
        # 확률이 0.3(30%) 이상이면 무조건 '발생(1)'으로 판정
        y_pred = (y_prob >= 0.3).astype(int)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] 
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 최종 검증: SMOTETomek & 조기경보 임계값(30%) 적용 결과 🔥")
print("   * 참고: 30%의 확률만 보여도 '발생'으로 예측하여 재현율을 극한으로 끌어올린 세팅입니다.")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) 
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 요건을 충족하는 데이터가 없습니다.")
print("-" * 105)

⏳ [마스터 레벨] SMOTETomek & 임계값 30% 방어 모델 스캔 중...

🔥 최종 검증: SMOTETomek & 조기경보 임계값(30%) 적용 결과 🔥
   * 참고: 30%의 확률만 보여도 '발생'으로 예측하여 재현율을 극한으로 끌어올린 세팅입니다.
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도     미발생(0) 방어력      발생(1) 적중률
   깨씨무늬병    373  LR 43.8%  14.9% (11/74) 100.0% (38/38)
   깨씨무늬병    373  RF 54.5%  35.1% (26/74)  92.1% (35/38)
   깨씨무늬병    373 KNN 50.0%  33.8% (25/74)  81.6% (31/38)
   깨씨무늬병    373 XGB 62.5%  62.2% (46/74)  63.2% (24/38)
   끝동매미충    362  LR 54.1% 54.0% (54/100)    55.6% (5/9)
   끝동매미충    362 KNN 54.1% 55.0% (55/100)    44.4% (4/9)
   끝동매미충    362  RF 68.8% 71.0% (71/100)    44.4% (4/9)
   끝동매미충    362 XGB 83.5% 87.0% (87/100)    44.4% (4/9)
    먹노린재    383  LR 47.0%    9.7% (6/62)  90.6% (48/53)
    먹노린재    383  RF 55.7%  29.0% (18/62)  86.8% (46/53)
    먹노린재    383 KNN 53.9%  35.5% (22/62)  75.5% (40/53)
    먹노린재    383 XGB 52.2%  40.3% (25/62)  66.0% (35/53)
     벼멸구    365  LR

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]
pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(14, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(14, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(14, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (테스트 40% + 이진 분류)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [가혹 조건] 테스트 세트 40% 기반 모델 스캔 중... (진짜 실력 검증)\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    
    # 테스트 비중이 40%로 커졌으므로, 최소 데이터 요구량을 8개로 늘려 분할 에러를 방지합니다.
    if len(class_counts) < 2 or class_counts.min() < 8: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    # 🌟 [변경점] test_size를 0.4 (40%)로 설정하여 검증을 극대화
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        
        # 임계값 30% 적용 (발생 확률이 30%만 넘어도 '발생'으로 판정)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_prob >= 0.3).astype(int)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] 
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 냉혹한 검증 완료: 테스트 데이터 40% 기반 실제 방어력 🔥")
print("   * 참고: 테스트 비중이 높아져 정확도는 하락하지만, 이것이 팩트에 가장 가까운 수치입니다.")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) 
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 8개 이상)을 충족하는 병해충이 없습니다.")
print("-" * 105)

⏳ [가혹 조건] 테스트 세트 40% 기반 모델 스캔 중... (진짜 실력 검증)

🔥 냉혹한 검증 완료: 테스트 데이터 40% 기반 실제 방어력 🔥
   * 참고: 테스트 비중이 높아져 정확도는 하락하지만, 이것이 팩트에 가장 가까운 수치입니다.
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도      미발생(0) 방어력      발생(1) 적중률
   깨씨무늬병    373  LR 50.7%   29.3% (29/99)  92.2% (47/51)
   깨씨무늬병    373 KNN 49.3%   32.3% (32/99)  82.4% (42/51)
   깨씨무늬병    373  RF 53.3%   39.4% (39/99)  80.4% (41/51)
   깨씨무늬병    373 XGB 61.3%   60.6% (60/99)  62.7% (32/51)
   끝동매미충    362  LR 51.0%  50.4% (67/133)   58.3% (7/12)
   끝동매미충    362 KNN 53.8%  55.6% (74/133)   33.3% (4/12)
   끝동매미충    362  RF 75.2% 79.7% (106/133)   25.0% (3/12)
   끝동매미충    362 XGB 77.9% 83.5% (111/133)   16.7% (2/12)
    먹노린재    383  RF 55.2%   26.2% (22/84)  90.0% (63/70)
    먹노린재    383  LR 50.0%   19.0% (16/84)  87.1% (61/70)
    먹노린재    383 KNN 55.8%   35.7% (30/84)  80.0% (56/70)
    먹노린재    383 XGB 57.1%   48.8% (41/84)  67.1% (47/70)
     벼멸구    36

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (14일 -> 7일로 변경하여 단기 민감도 극대화)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

# 모든 rolling 기준을 7로 수정
weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

# 연속 강수일수도 7일 기준으로 변경
weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [7일 피처] 단기 기상 패턴 기반 조기경보 모델 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        
        # 임계값 30% 적용
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_prob >= 0.3).astype(int)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] 
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 7일 압축 피처 + 조기경보 임계값(30%) 방제 시스템 검증 결과 🔥")
print("-" * 105)
if len(results_df) > 0:
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) 
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건을 충족하는 데이터가 없습니다.")
print("-" * 105)

⏳ [7일 피처] 단기 기상 패턴 기반 조기경보 모델 스캔 중...

🔥 7일 압축 피처 + 조기경보 임계값(30%) 방제 시스템 검증 결과 🔥
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도     미발생(0) 방어력     발생(1) 적중률
   깨씨무늬병    373  LR 42.9%  14.9% (11/74) 97.4% (37/38)
   깨씨무늬병    373  RF 51.8%  29.7% (22/74) 94.7% (36/38)
   깨씨무늬병    373 KNN 55.4%  41.9% (31/74) 81.6% (31/38)
   깨씨무늬병    373 XGB 59.8%  59.5% (44/74) 60.5% (23/38)
   끝동매미충    362  LR 41.3% 41.0% (41/100)   44.4% (4/9)
   끝동매미충    362 KNN 48.6% 49.0% (49/100)   44.4% (4/9)
   끝동매미충    362  RF 68.8% 71.0% (71/100)   44.4% (4/9)
   끝동매미충    362 XGB 77.1% 80.0% (80/100)   44.4% (4/9)
    먹노린재    383  LR 49.6%    8.1% (5/62) 98.1% (52/53)
    먹노린재    383  RF 49.6%  22.6% (14/62) 81.1% (43/53)
    먹노린재    383 KNN 53.0%  32.3% (20/62) 77.4% (41/53)
    먹노린재    383 XGB 53.9%  54.8% (34/62) 52.8% (28/53)
     벼멸구    365  LR 30.0%  15.7% (14/89) 90.5% (19/21)
     벼멸구    365 KNN 50.9%  46.1% (41/89) 71

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (직전 7일 기준, 11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (테스트 30% + 표준 임계값 50%)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [표준 검증] 테스트 30% 기반 균형 잡힌 모델 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTETomek 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        
        # 🌟 표준 50% 임계값 적용 (기본 예측 방식 사용)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] 
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 표준 검증 완료: 테스트 30% + 임계값 50% 균형 평가 결과 🔥")
print("-" * 105)
if len(results_df) > 0:
    # 발생(1) 재현율이 높은 순서대로 정렬
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) 
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 5개 이상)을 충족하는 데이터가 없습니다.")
print("-" * 105)

⏳ [표준 검증] 테스트 30% 기반 균형 잡힌 모델 스캔 중...

🔥 표준 검증 완료: 테스트 30% + 임계값 50% 균형 평가 결과 🔥
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도      미발생(0) 방어력     발생(1) 적중률
   깨씨무늬병    373  LR 67.0%   62.2% (46/74) 76.3% (29/38)
   깨씨무늬병    373 KNN 58.9%   52.7% (39/74) 71.1% (27/38)
   깨씨무늬병    373  RF 59.8%   60.8% (45/74) 57.9% (22/38)
   깨씨무늬병    373 XGB 57.1%   64.9% (48/74) 42.1% (16/38)
   끝동매미충    362  LR 63.3%  65.0% (65/100)   44.4% (4/9)
   끝동매미충    362 KNN 56.9%  58.0% (58/100)   44.4% (4/9)
   끝동매미충    362 XGB 79.8%  83.0% (83/100)   44.4% (4/9)
   끝동매미충    362  RF 81.7%  86.0% (86/100)   33.3% (3/9)
    먹노린재    383 KNN 55.7%   56.5% (35/62) 54.7% (29/53)
    먹노린재    383  LR 57.4%   66.1% (41/62) 47.2% (25/53)
    먹노린재    383 XGB 55.7%   67.7% (42/62) 41.5% (22/53)
    먹노린재    383  RF 57.4%   72.6% (45/62) 39.6% (21/53)
     벼멸구    365 KNN 61.8%   64.0% (57/89) 52.4% (11/21)
     벼멸구    365  LR 52.7%   53

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (직전 7일 기준, 11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 모든 병해충 모델 스캔 (테스트 30% + 임계값 40%)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

print("⏳ [균형점 탐색] 테스트 30% & 발생 임계값 40% 스캔 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTETomek 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    models = {
        "LR": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced'),
        "XGB": XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio)
    }
    
    clean_name = target.replace(' (피해율)', '')
    
    for model_name, model in models.items():
        model.fit(X_train_resampled, y_train_resampled)
        
        # 🌟 타협점 40% 임계값 적용 (발생 확률이 40%만 넘어도 '발생'으로 판정)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_prob >= 0.4).astype(int)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '모델': model_name,
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어력': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중률': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})",
            '_sort_val': recalls[1] 
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 최적화 검증 완료: 테스트 30% + 임계값 40% 타협점 결과 🔥")
print("-" * 105)
if len(results_df) > 0:
    # 발생(1) 재현율이 높은 순서대로 정렬
    results_df = results_df.sort_values(by=['병해충명', '_sort_val'], ascending=[True, False])
    results_df = results_df.drop(columns=['_sort_val']) 
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건(0, 1 등급 데이터가 각각 최소 5개 이상)을 충족하는 데이터가 없습니다.")
print("-" * 105)

⏳ [균형점 탐색] 테스트 30% & 발생 임계값 40% 스캔 중...

🔥 최적화 검증 완료: 테스트 30% + 임계값 40% 타협점 결과 🔥
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터  모델   정확도     미발생(0) 방어력     발생(1) 적중률
   깨씨무늬병    373  LR 58.0%  39.2% (29/74) 94.7% (36/38)
   깨씨무늬병    373 KNN 55.4%  41.9% (31/74) 81.6% (31/38)
   깨씨무늬병    373  RF 55.4%  43.2% (32/74) 78.9% (30/38)
   깨씨무늬병    373 XGB 57.1%  62.2% (46/74) 47.4% (18/38)
   끝동매미충    362  LR 51.4% 52.0% (52/100)   44.4% (4/9)
   끝동매미충    362 KNN 48.6% 49.0% (49/100)   44.4% (4/9)
   끝동매미충    362 XGB 78.0% 81.0% (81/100)   44.4% (4/9)
   끝동매미충    362  RF 77.1% 81.0% (81/100)   33.3% (3/9)
    먹노린재    383 KNN 53.0%  32.3% (20/62) 77.4% (41/53)
    먹노린재    383  LR 47.0%  33.9% (21/62) 62.3% (33/53)
    먹노린재    383  RF 54.8%  48.4% (30/62) 62.3% (33/53)
    먹노린재    383 XGB 54.8%  59.7% (37/62) 49.1% (26/53)
     벼멸구    365  LR 45.5%  37.1% (33/89) 81.0% (17/21)
     벼멸구    365 KNN 50.9%  46.1% (41/89) 71

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 6월 제외 (7, 8, 9월 데이터만 사용)
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]

pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 🌟 기상 데이터 피처 생성 (직전 7일 기준, 11개 핵심 피처)
# ==========================================================
weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 🌟 XGBoost 전용 스캔 (가중치 배수별 성능 테스트)
# ==========================================================
target_diseases = [col for col in pest_df.columns if '(피해율)' in col]
all_results = []

# 테스트해 볼 가중치 배수 (기본 1배, 2배, 3배, 5배)
weight_multipliers = [1, 2, 3, 5]

print("⏳ [XGBoost 심화 튜닝] 페널티 가중치(1배~5배) 시뮬레이션 중...\n")

for target in target_diseases:
    df_subset = pest_df[['조사일', '매칭지역', target]].dropna()
    if len(df_subset) == 0: continue
        
    df_subset['target_bin'] = (df_subset[target] > 0).astype(int)
    merged_df = pd.merge(df_subset, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner').dropna()
    
    class_counts = merged_df['target_bin'].value_counts()
    if len(class_counts) < 2 or class_counts.min() < 5: 
        continue
        
    X = merged_df[feature_list]
    y = merged_df['target_bin']
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
        
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # SMOTETomek 적용
    min_class_count = y_train.value_counts().min()
    k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1
    
    if min_class_count > 1:
        smote_base = SMOTE(k_neighbors=k_neighbors, random_state=42)
        try:
            resampler = SMOTETomek(smote=smote_base, random_state=42)
            X_train_resampled, y_train_resampled = resampler.fit_resample(X_train_scaled, y_train)
        except:
            X_train_resampled, y_train_resampled = smote_base.fit_resample(X_train_scaled, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train_scaled, y_train
        
    # 기본 비율 계산 (0의 개수 / 1의 개수)
    base_ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1) if np.sum(y_train == 1) > 0 else 1
    
    clean_name = target.replace(' (피해율)', '')
    
    # 🌟 각 배수(Multiplier)별로 XGBoost를 학습시키고 평가
    for m in weight_multipliers:
        current_weight = base_ratio * m
        
        xgb_model = XGBClassifier(
            random_state=42, 
            eval_metric='logloss', 
            scale_pos_weight=current_weight  # 페널티 강제 주입!
        )
        
        xgb_model.fit(X_train_resampled, y_train_resampled)
        
        # 임계값 40% 적용
        y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_prob >= 0.4).astype(int)
        
        acc = accuracy_score(y_test, y_pred) * 100
        recalls = recall_score(y_test, y_pred, labels=[0, 1], average=None, zero_division=0) * 100
        
        actual_0 = sum(y_test == 0)
        actual_1 = sum(y_test == 1)
        correct_0 = sum((y_test == 0) & (y_pred == 0))
        correct_1 = sum((y_test == 1) & (y_pred == 1))
        
        all_results.append({
            '병해충명': clean_name,
            '총 데이터': len(merged_df),
            '가중치': f"기본 x {m}배",
            '정확도': f"{acc:.1f}%",
            '미발생(0) 방어': f"{recalls[0]:.1f}% ({correct_0}/{actual_0})",
            '발생(1) 적중': f"{recalls[1]:.1f}% ({correct_1}/{actual_1})"
        })

# ==========================================================
# 🌟 결과 출력
# ==========================================================
results_df = pd.DataFrame(all_results)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("🔥 XGBoost 가중치 튜닝 결과 (임계값 40% 고정) 🔥")
print("   * 가중치 배수가 높아질수록 '발생(1)'을 더 악착같이 잡아냅니다.")
print("   * 하지만 너무 높아지면 다 '발생'이라고 우겨서 '미발생(0)' 방어력이 무너집니다.")
print("-" * 105)
if len(results_df) > 0:
    # 병해충명 기준으로 묶어서 가중치 변화 흐름을 쉽게 보도록 정렬
    results_df = results_df.sort_values(by=['병해충명', '가중치'], ascending=[True, True])
    print(results_df.to_string(index=False))
else:
    print("⚠️ 분석 가능한 요건을 충족하는 데이터가 없습니다.")
print("-" * 105)

⏳ [XGBoost 심화 튜닝] 페널티 가중치(1배~5배) 시뮬레이션 중...

🔥 XGBoost 가중치 튜닝 결과 (임계값 40% 고정) 🔥
   * 가중치 배수가 높아질수록 '발생(1)'을 더 악착같이 잡아냅니다.
   * 하지만 너무 높아지면 다 '발생'이라고 우겨서 '미발생(0)' 방어력이 무너집니다.
---------------------------------------------------------------------------------------------------------
    병해충명  총 데이터     가중치   정확도       미발생(0) 방어      발생(1) 적중
   깨씨무늬병    373 기본 x 1배 57.1%   62.2% (46/74) 47.4% (18/38)
   깨씨무늬병    373 기본 x 2배 58.9%   58.1% (43/74) 60.5% (23/38)
   깨씨무늬병    373 기본 x 3배 55.4%   54.1% (40/74) 57.9% (22/38)
   깨씨무늬병    373 기본 x 5배 53.6%   50.0% (37/74) 60.5% (23/38)
   끝동매미충    362 기본 x 1배 78.0%  81.0% (81/100)   44.4% (4/9)
   끝동매미충    362 기본 x 2배 76.1%  79.0% (79/100)   44.4% (4/9)
   끝동매미충    362 기본 x 3배 72.5%  75.0% (75/100)   44.4% (4/9)
   끝동매미충    362 기본 x 5배 74.3%  77.0% (77/100)   44.4% (4/9)
    먹노린재    383 기본 x 1배 54.8%   59.7% (37/62) 49.1% (26/53)
    먹노린재    383 기본 x 2배 51.3%   53.2% (33/62) 49.1% (26/53)
    먹노린재    383 기본 x 3배 53.0%   54.8% (34/62) 50.9% (27/53)


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# ==========================================================
# 1. 데이터 불러오기 및 기본 전처리
# ==========================================================
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 🌟 7, 8, 9월 집중 분석
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]
pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 2. 기상 데이터 피처 엔지니어링 (7일 압축 + 신규 변수)
# ==========================================================
# 💡 신규 변수 추가: '월' 정보와 '일교차'
weather_df['월'] = weather_df['일시'].dt.month
if '최고기온(°C)' in weather_df.columns and '최저기온(°C)' in weather_df.columns:
    weather_df['일교차'] = weather_df['최고기온(°C)'] - weather_df['최저기온(°C)']
else:
    weather_df['일교차'] = 0

weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

# 직전 7일 롤링(민감도 극대화)
weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

# 💡 월과 일교차가 포함된 총 13개의 막강한 피처 리스트
feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수', '월', '일교차']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 3. 데이터 구조 변경 (Wide -> Long Format)
# ==========================================================
pest_cols = [col for col in pest_df.columns if '(피해율)' in col]

# 세로로 길게 풀어서 2단계 분류가 가능하도록 재조립
melted_df = pest_df.melt(id_vars=['조사일', '매칭지역'], value_vars=pest_cols, var_name='Pest', value_name='Damage')
melted_df = melted_df.dropna(subset=['Damage'])
melted_df['Pest'] = melted_df['Pest'].str.replace(' (피해율)', '', regex=False)

# 기상 데이터와 최종 병합
full_df = pd.merge(melted_df, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner')

# ==========================================================
# 🎯 [STAGE 1] 통합 발생 여부 예측 (이진 분류)
# ==========================================================
print("⏳ STAGE 1: 병해충 발생 조기경보 모델 학습 중... (SMOTETomek 적용)")

# 특정 날짜/지역에 병해충이 '하나라도' 발생했는지 통합
stage1_df = full_df.groupby(['조사일', '매칭지역'] + feature_list)['Damage'].max().reset_index()
stage1_df['Is_Occurred'] = (stage1_df['Damage'] > 0).astype(int)

X1 = stage1_df[feature_list]
y1 = stage1_df['Is_Occurred']

# 테스트 세트 30% 보장
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.3, random_state=42, stratify=y1)

scaler1 = StandardScaler()
X1_train_scaled = scaler1.fit_transform(X1_train)
X1_test_scaled = scaler1.transform(X1_test)

# SMOTETomek으로 경계선 노이즈 정리 및 데이터 증강
try:
    smote_tomek = SMOTETomek(random_state=42)
    X1_train_res, y1_train_res = smote_tomek.fit_resample(X1_train_scaled, y1_train)
except:
    X1_train_res, y1_train_res = X1_train_scaled, y1_train

# 1단계 모델: Random Forest (과적합에 강함)
model_stage1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_stage1.fit(X1_train_res, y1_train_res)

# 🌟 임계값 40% 세팅 (발생 확률이 40%만 넘어도 '발생'으로 간주하여 재현율 확보)
y1_prob = model_stage1.predict_proba(X1_test_scaled)[:, 1]
y1_pred = (y1_prob >= 0.4).astype(int)

acc1 = accuracy_score(y1_test, y1_pred)
rec1 = recall_score(y1_test, y1_pred)
prec1 = precision_score(y1_test, y1_pred, zero_division=0)

print("\n" + "="*70)
print(f"✔️ [STAGE 1] 발생/미발생 예측 결과 (임계값 40%)")
print("="*70)
print(f" - 정확도 (Accuracy)  : {acc1*100:5.1f}% (전체 예측 성공률)")
print(f" - 재현율 (Recall)    : {rec1*100:5.1f}% (실제 발생 건수 중 경보를 울린 비율)")
print(f" - 정밀도 (Precision) : {prec1*100:5.1f}% (경보를 울렸을 때 진짜 발생한 비율)")

# ==========================================================
# 🎯 [STAGE 2] 발생 건 대상 병해충 종류 판별 (다중 분류)
# ==========================================================
print("\n⏳ STAGE 2: 발생 건 대상 병해충 종류 판별 모델 학습 중... (XGBoost)")

# 1단계에서 발생(Damage > 0)한 데이터만 모아서 종류 맞추기
stage2_df = full_df[full_df['Damage'] > 0]
# 같은 날 여러 개가 발생했다면 가장 피해율이 큰(대표) 해충 하나만 남김
stage2_df = stage2_df.sort_values('Damage', ascending=False).drop_duplicates(['조사일', '매칭지역'])

# 모델 학습을 위해 너무 희귀한 병해충(10건 미만)은 제거
pest_counts = stage2_df['Pest'].value_counts()
valid_pests = pest_counts[pest_counts >= 10].index
stage2_df = stage2_df[stage2_df['Pest'].isin(valid_pests)]

X2 = stage2_df[feature_list]
y2_raw = stage2_df['Pest']

# 문자로 된 병해충 이름을 숫자로 인코딩 (0, 1, 2...)
le = LabelEncoder()
y2 = le.fit_transform(y2_raw)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

# 2단계 모델: XGBoost 다중 분류 (멀티클래스 문제에 강력함)
model_stage2 = XGBClassifier(random_state=42, objective='multi:softmax', num_class=len(valid_pests))
model_stage2.fit(X2_train_scaled, y2_train) 

y2_pred = model_stage2.predict(X2_test_scaled)

# 평가 지표 산출 (Macro: 모든 병해충을 동등한 비중으로 평가)
acc2 = accuracy_score(y2_test, y2_pred)
rec2 = recall_score(y2_test, y2_pred, average='macro', zero_division=0)
prec2 = precision_score(y2_test, y2_pred, average='macro', zero_division=0)

print("\n" + "="*70)
print(f"✔️ [STAGE 2] 병해충 종류 판별 결과 (대상: {len(valid_pests)}종)")
print("="*70)
print(f" - 전체 정확도 (Accuracy) : {acc2*100:5.1f}%")
print(f" - 평균 재현율 (Macro Rec): {rec2*100:5.1f}%")
print(f" - 평균 정밀도 (Macro Pre): {prec2*100:5.1f}%\n")

print("🔍 [병해충별 상세 성적표]")
for i, pest_name in enumerate(le.classes_):
    idx = (y2_test == i)
    if sum(idx) == 0: continue
    
    pest_rec = recall_score(y2_test == i, y2_pred == i, zero_division=0)
    pest_prec = precision_score(y2_test == i, y2_pred == i, zero_division=0)
    
    print(f" ▶ {pest_name:<12}: 재현율 {pest_rec*100:5.1f}% | 정밀도 {pest_prec*100:5.1f}% (테스트 {sum(idx)}건)")
print("="*70)

⏳ STAGE 1: 병해충 발생 조기경보 모델 학습 중... (SMOTETomek 적용)

✔️ [STAGE 1] 발생/미발생 예측 결과 (임계값 40%)
 - 정확도 (Accuracy)  :  86.2% (전체 예측 성공률)
 - 재현율 (Recall)    :  96.0% (실제 발생 건수 중 경보를 울린 비율)
 - 정밀도 (Precision) :  88.9% (경보를 울렸을 때 진짜 발생한 비율)

⏳ STAGE 2: 발생 건 대상 병해충 종류 판별 모델 학습 중... (XGBoost)

✔️ [STAGE 2] 병해충 종류 판별 결과 (대상: 7종)
 - 전체 정확도 (Accuracy) :  36.2%
 - 평균 재현율 (Macro Rec):  28.8%
 - 평균 정밀도 (Macro Pre):  33.5%

🔍 [병해충별 상세 성적표]
 ▶ 깨씨무늬병       : 재현율  33.3% | 정밀도 100.0% (테스트 6건)
 ▶ 먹노린재        : 재현율  57.9% | 정밀도  42.3% (테스트 19건)
 ▶ 벼멸구         : 재현율  50.0% | 정밀도  28.6% (테스트 4건)
 ▶ 벼물바구미       : 재현율   0.0% | 정밀도   0.0% (테스트 3건)
 ▶ 잎집무늬마름병     : 재현율  45.9% | 정밀도  50.0% (테스트 37건)
 ▶ 혹명나방        : 재현율  14.3% | 정밀도  13.3% (테스트 14건)
 ▶ 흰등멸구        : 재현율   0.0% | 정밀도   0.0% (테스트 11건)


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기
try:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='utf-8-sig')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='utf-8-sig')
except:
    pest_df = pd.read_csv("통합_논벼_예찰결과_피해율유지.csv", encoding='cp949')
    weather_df = pd.read_csv("전남_일별_기상청_데이터_최종수정.csv", encoding='cp949')

pest_df['조사일'] = pd.to_datetime(pest_df['조사일'].astype(str).str.replace('.', '-'))
weather_df['일시'] = pd.to_datetime(weather_df['일시'])

# 7, 8, 9월만 사용
pest_df = pest_df[pest_df['조사일'].dt.month.isin([7, 8, 9])]
pest_df['매칭지역'] = pest_df['지역'].astype(str).str.replace(r'[시군]$', '', regex=True)
weather_df['매칭지역'] = weather_df['지점명'].astype(str)

# ==========================================================
# 2. 기상 데이터 피처 생성 (13개 핵심 피처)
# ==========================================================
weather_df['월'] = weather_df['일시'].dt.month
if '최고기온(°C)' in weather_df.columns and '최저기온(°C)' in weather_df.columns:
    weather_df['일교차'] = weather_df['최고기온(°C)'] - weather_df['최저기온(°C)']
else:
    weather_df['일교차'] = 0

weather_df = weather_df.sort_values(['매칭지역', '일시']).reset_index(drop=True)
wg = weather_df.groupby('매칭지역')

weather_df['평균기온_mean']   = wg['평균기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_mean']   = wg['최고기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최저기온_mean']   = wg['최저기온(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['습도_mean']       = wg['평균 상대습도(%)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['강수량_mean']     = wg['일강수량(mm)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['일조시간_mean']   = wg['합계 일조시간(hr)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['평균풍속_mean']   = wg['평균 풍속(m/s)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['이슬점온도_mean'] = wg['평균 이슬점온도(°C)'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
weather_df['최고기온_max']    = wg['최고기온(°C)'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
weather_df['최저기온_min']    = wg['최저기온(°C)'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

def get_max_consecutive_rain(series):
    is_rainy = series > 0
    if not is_rainy.any(): return 0
    return is_rainy.groupby((~is_rainy).cumsum()).sum().max()

weather_df['연속강수일수'] = wg['일강수량(mm)'].rolling(7, min_periods=1).apply(get_max_consecutive_rain, raw=False).reset_index(level=0, drop=True)

feature_list = ['평균기온_mean', '최고기온_mean', '최저기온_mean', '습도_mean', 
                '강수량_mean', '일조시간_mean', '평균풍속_mean', '이슬점온도_mean',
                '최고기온_max', '최저기온_min', '연속강수일수', '월', '일교차']

w_features = weather_df[['매칭지역', '일시'] + feature_list]

# ==========================================================
# 3. 데이터 구조 변경 (Wide -> Long Format)
# ==========================================================
pest_cols = [col for col in pest_df.columns if '(피해율)' in col]
melted_df = pest_df.melt(id_vars=['조사일', '매칭지역'], value_vars=pest_cols, var_name='Pest', value_name='Damage')
melted_df = melted_df.dropna(subset=['Damage'])
melted_df['Pest'] = melted_df['Pest'].str.replace(' (피해율)', '', regex=False)

full_df = pd.merge(melted_df, w_features, left_on=['조사일', '매칭지역'], right_on=['일시', '매칭지역'], how='inner')

# ==========================================================
# 🎯 [STAGE 1] 통합 발생 여부 (생략 없이 실행)
# ==========================================================
print("⏳ STAGE 1: 발생 여부 조기경보 모델 학습 중...")
stage1_df = full_df.groupby(['조사일', '매칭지역'] + feature_list)['Damage'].max().reset_index()
stage1_df['Is_Occurred'] = (stage1_df['Damage'] > 0).astype(int)

X1 = stage1_df[feature_list]
y1 = stage1_df['Is_Occurred']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.3, random_state=42, stratify=y1)

scaler1 = StandardScaler()
X1_train_scaled = scaler1.fit_transform(X1_train)
X1_test_scaled = scaler1.transform(X1_test)

try:
    smote_tomek = SMOTETomek(random_state=42)
    X1_train_res, y1_train_res = smote_tomek.fit_resample(X1_train_scaled, y1_train)
except:
    X1_train_res, y1_train_res = X1_train_scaled, y1_train

model_stage1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_stage1.fit(X1_train_res, y1_train_res)

y1_prob = model_stage1.predict_proba(X1_test_scaled)[:, 1]
y1_pred = (y1_prob >= 0.4).astype(int)

acc1 = accuracy_score(y1_test, y1_pred)
rec1 = recall_score(y1_test, y1_pred)
prec1 = precision_score(y1_test, y1_pred, zero_division=0)

print("\n" + "="*70)
print(f"✔️ [STAGE 1] 발생/미발생 (임계값 40%) : 재현율 {rec1*100:.1f}% | 정밀도 {prec1*100:.1f}%")
print("="*70)

# ==========================================================
# 🎯 [STAGE 2] Any-Hit 복합 채점 방식 도입 (🔥핵심 변경 구간)
# ==========================================================
print("\n⏳ STAGE 2: 복합 발생 허용(Any-Hit) 해충 분류 모델 학습 중...")

# 1. 발생한 데이터 필터링
stage2_full = full_df[full_df['Damage'] > 0]

# 💡 [핵심 아이디어] 그날 그 지역에 발생한 '모든' 해충 리스트를 묶어둔다. (정답지 리스트)
actual_pests_agg = stage2_full.groupby(['조사일', '매칭지역'])['Pest'].apply(list).reset_index(name='Actual_Pest_List')

# 2. 학습을 위해서는 가장 피해가 큰 '주요 해충(Dominant Pest)' 1개를 타겟으로 삼음
stage2_train_df = stage2_full.sort_values('Damage', ascending=False).drop_duplicates(['조사일', '매칭지역'])

# 리스트 정답지와 주요 해충 데이터를 병합
stage2_df = pd.merge(stage2_train_df, actual_pests_agg, on=['조사일', '매칭지역'])

pest_counts = stage2_df['Pest'].value_counts()
valid_pests = pest_counts[pest_counts >= 10].index
stage2_df = stage2_df[stage2_df['Pest'].isin(valid_pests)]

X2 = stage2_df[feature_list]
y2_raw = stage2_df['Pest']
y2_actual_lists = stage2_df['Actual_Pest_List'] # 복합 정답지

le = LabelEncoder()
y2 = le.fit_transform(y2_raw)

# X(피처), y(학습용 단일타겟), 그리고 y_list(평가용 다중정답지)를 동시에 쪼갬
X2_train, X2_test, y2_train, y2_test, _, y2_test_lists = train_test_split(
    X2, y2, y2_actual_lists, test_size=0.3, random_state=42, stratify=y2
)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

model_stage2 = XGBClassifier(random_state=42, objective='multi:softmax', num_class=len(valid_pests))
model_stage2.fit(X2_train_scaled, y2_train) 

# 예측 수행 (숫자로 반환됨)
y2_pred_numeric = model_stage2.predict(X2_test_scaled)
# 숫자를 다시 해충 이름(문자열)으로 변환
y2_pred_names = le.inverse_transform(y2_pred_numeric)

# 💡 [혁신 평가 로직] Any-Hit Accuracy 계산
hit_count = 0
total_count = len(y2_test_lists)

for pred_name, actual_list in zip(y2_pred_names, y2_test_lists):
    # 모델이 예측한 1개의 해충이, 실제 발생한 해충 리스트 안에 들어있다면 정답 처리!
    if pred_name in actual_list:
        hit_count += 1

any_hit_accuracy = (hit_count / total_count) * 100

print("\n" + "="*70)
print(f"✔️ [STAGE 2] 병해충 종류 실무형 채점 결과 (Any-Hit Accuracy)")
print("   * 모델이 예측한 해충이 그날 발생한 여러 해충 중 하나라도 일치하면 정답")
print("="*70)
print(f" 🎯 복합 정답률 (Any-Hit Score) : {any_hit_accuracy:.1f}% ({hit_count}건 성공 / 총 {total_count}건)")
print("="*70)

# 기존의 엄격한(Strict) 정확도와 비교
strict_acc = accuracy_score(y2_test, y2_pred_numeric) * 100
print(f" (참고) 가장 심한 해충 1개만 맞추는 기존 엄격한 정확도 : {strict_acc:.1f}%")

⏳ STAGE 1: 발생 여부 조기경보 모델 학습 중...

✔️ [STAGE 1] 발생/미발생 (임계값 40%) : 재현율 96.0% | 정밀도 88.9%

⏳ STAGE 2: 복합 발생 허용(Any-Hit) 해충 분류 모델 학습 중...

✔️ [STAGE 2] 병해충 종류 실무형 채점 결과 (Any-Hit Accuracy)
   * 모델이 예측한 해충이 그날 발생한 여러 해충 중 하나라도 일치하면 정답
 🎯 복합 정답률 (Any-Hit Score) : 79.8% (75건 성공 / 총 94건)
 (참고) 가장 심한 해충 1개만 맞추는 기존 엄격한 정확도 : 36.2%
